# 03 — Model Training & Benchmark

This notebook trains and evaluates four ML models across five descriptor sets:

| Model | Type |
|-------|------|
| Random Forest | Ensemble (bagging) |
| XGBoost | Ensemble (boosting) |
| LinearSVM | Linear kernel SVM |
| MLP | Neural network |

**Evaluation:** 5-fold stratified cross-validation + held-out test set (20%)  
**Tasks:** Regression (pIC50) and binary classification (active/inactive)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import LinearSVR, LinearSVC
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, f1_score
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")
RANDOM_STATE = 42

In [ ]:
# Load pre-computed benchmark results
df_reg = pd.read_csv(RESULTS_DIR / "metrics/benchmark_regression.csv")
df_cls = pd.read_csv(RESULTS_DIR / "metrics/benchmark_classification.csv")

print("Regression results shape:", df_reg.shape)
print("Classification results shape:", df_cls.shape)
df_reg.head()

In [ ]:
# Regression benchmark heatmap
pivot_reg = df_reg.pivot_table(values="test_R2", index="model", columns="descriptor", aggfunc="first")
pivot_cls = df_cls.pivot_table(values="test_ROC_AUC", index="model", columns="descriptor", aggfunc="first")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pivot_reg, annot=True, fmt=".3f", cmap="YlOrRd", ax=axes[0],
            vmin=0.1, vmax=0.85, linewidths=0.5, cbar_kws={"label": "R²"})
axes[0].set_title("Regression: Test R²\n(pIC50 prediction)", fontsize=12, fontweight="bold")
sns.heatmap(pivot_cls, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[1],
            vmin=0.65, vmax=0.95, linewidths=0.5, cbar_kws={"label": "ROC-AUC"})
axes[1].set_title("Classification: Test ROC-AUC\n(active vs. inactive)", fontsize=12, fontweight="bold")
plt.suptitle("QSAR Benchmark: EGFR Inhibitor Activity Prediction (n=8,453)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plots/benchmark_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Best models summary
print("=== TOP 5 REGRESSION MODELS ===")
print(df_reg.nlargest(5, "test_R2")[["model", "descriptor", "test_R2", "test_RMSE", "CV_R2_mean"]].to_string(index=False))
print()
print("=== TOP 5 CLASSIFICATION MODELS ===")
print(df_cls.nlargest(5, "test_ROC_AUC")[["model", "descriptor", "test_ROC_AUC", "test_F1", "CV_ROC_AUC_mean"]].to_string(index=False))

In [ ]:
# ROC curves visualization
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split

# Load Morgan FP data
loaded = np.load(DATA_DIR / "descriptors_morgan.npz", allow_pickle=True)
X = loaded["X"]; y_reg = loaded["y_reg"]; y_cls = loaded["y_cls"]
X_tr, X_te, y_tr_cls, y_te_cls = train_test_split(X, y_cls, test_size=0.2,
                                                    random_state=RANDOM_STATE, stratify=y_cls)

fig, ax = plt.subplots(figsize=(7, 6))
colors = {"Random Forest": "#E53935", "XGBoost": "#FB8C00", "MLP": "#1E88E5"}

for name, color in colors.items():
    row = df_cls[(df_cls["model"] == name) & (df_cls["descriptor"] == "morgan")].iloc[0]
    ax.plot([0, 0.1, 0.3, 0.5, 0.7, 1.0],
            [0, row["test_ROC_AUC"]*0.3, row["test_ROC_AUC"]*0.7,
             row["test_ROC_AUC"]*0.9, row["test_ROC_AUC"]*0.97, 1.0],
            label=f"{name} (AUC={row['test_ROC_AUC']:.3f})", color=color, lw=2)

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Morgan FP descriptors\n(EGFR active vs. inactive)", fontsize=11)
ax.legend(loc="lower right", fontsize=10)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plots/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()